In [1]:
import cupy as cp
import numpy as np
import pandas as pd

#Making sure a GPU is assigned
print(f"Current device name: {cp.cuda.runtime.getDeviceProperties(0)["name"]}")
mempool = cp.get_default_memory_pool()

Current device name: b'NVIDIA GeForce RTX 3070'


In [2]:
policies = pd.read_csv(r"C:\Users\pauln\OneDrive\Code\Parallelization\POLICIES.csv")
scenarios = pd.read_csv(r"C:\Users\pauln\OneDrive\Code\Parallelization\RETURN.csv")

OUTER_YEARS = 100
OUTER_SCENS = 500
NB_POLICIES = 500
NB_CALCS = 16
this_dtype = 'float64'

_T = 0
_S = 1
_POLICY = 2
_MV = 3
_RETURN = 4
_TRANSAC = 5
_GUARANTEE = 6
_CLAIMS = 0
_CALC1 = 1
_CALC2 = 2
_CALC3 = 3
_CALC4 = 4
_CALC5 = 5
_CALC6 = 6
_CALC7 = 7
_CALC8 = 8
_CALC9 = 9
_CALC10 = 10
_CALC11 = 11
_CALC12 = 12
_CALC13 = 13
_CALC14 = 14
_CALC15 = 15

OL_df = pd.merge(policies[policies.POLICY <= NB_POLICIES],scenarios,how='cross')
OL_df = OL_df[['T','S','POLICY','MV','RETURN','TRANSAC','GUARANTEE']]
#OL_df[['CLAIMS','CALC1','CALC2','CALC3','CALC4','CALC5','CALC6','CALC7','CALC8','CALC9','CALC10','CALC11','CALC12','CALC13','CALC14','CALC15']] = 0
OL_df = OL_df.sort_values(['T','S','POLICY'])

Load Numpy

In [3]:
OL_np = OL_df.to_numpy(dtype=this_dtype).reshape(OUTER_YEARS,OUTER_SCENS,NB_POLICIES,OL_df.shape[1])
OC_np = np.zeros(OL_np.shape[:-1] + (NB_CALCS,),dtype=this_dtype)

print(f"OuterLoop shape      : {OL_np.shape}")
print(f"OuterLoop memory (GB): {OL_np.nbytes/1e9}")

OuterLoop shape      : (100, 500, 500, 7)
OuterLoop memory (GB): 1.4


Loop Numpy

In [4]:
for T in range(1,OUTER_YEARS-1):
    OL_np[T][...,_MV] = OL_np[T-1][...,_MV] * ( 1 + OL_np[T][...,_RETURN]) + OL_np[T][...,_TRANSAC]
    OL_np[T][...,_GUARANTEE] = OL_np[T-1][...,_GUARANTEE]

    OC_np[T][...,_CALC1] = (OL_np[T][...,_MV] + OL_np[T][...,_GUARANTEE])/2
    OC_np[T][...,_CALC2] = (OL_np[T][...,_MV]/1000)**2
    OC_np[T][...,_CALC3] = OL_np[T][...,_MV]*( 1 + OL_np[T][...,_RETURN])**1.5
    OC_np[T][...,_CALC4] = np.nan_to_num(OL_np[T][...,_TRANSAC]/OL_np[T][...,_MV])
    OC_np[T][...,_CALC5] = 1 - ( 1 + OL_np[T][...,_RETURN])**0.5
    OC_np[T][...,_CALC6] = (OL_np[T][...,_MV] - OL_np[T][...,_GUARANTEE])/2
    OC_np[T][...,_CALC7] = (-OL_np[T][...,_MV]/1000)**2
    OC_np[T][...,_CALC8] = OL_np[T][...,_MV]*( 1 - OL_np[T][...,_RETURN])**1.5
    OC_np[T][...,_CALC9] = np.nan_to_num(OL_np[T][...,_TRANSAC]/OL_np[T][...,_MV])
    OC_np[T][...,_CALC10] = 1 - ( 1 - OL_np[T][...,_RETURN])**0.5
    OC_np[T][...,_CALC11] = (OL_np[T][...,_MV] - OL_np[T][...,_GUARANTEE])/2
    OC_np[T][...,_CALC12] = (-OL_np[T][...,_MV]/1000)**2
    OC_np[T][...,_CALC13] = OL_np[T][...,_MV]*( 1 - OL_np[T][...,_RETURN])**1.5
    OC_np[T][...,_CALC14] = np.nan_to_num(OL_np[T][...,_TRANSAC]/OL_np[T][...,_MV])
    OC_np[T][...,_CALC15] = 1 - ( 1 - OL_np[T][...,_RETURN])**0.5

    if T % 10 == 0:
        OL_np[T][...,_GUARANTEE] = np.maximum(OL_np[T-1][...,_GUARANTEE], OL_np[T][...,_MV])

    OC_np[T][...,_CLAIMS] = np.maximum((OL_np[T][...,_GUARANTEE] - OL_np[T][...,_MV]),0)


Summarize Numpy

In [5]:
NP_result = np.nansum(np.nanmean(np.nansum(OC_np[...,_CLAIMS],axis=0),axis=0),axis=0)

print(NP_result)

1446877381.6409707


Load Cupy

In [6]:
OL_cp = cp.asarray(OL_np)
OC_cp = cp.zeros(OL_cp.shape[:-1] + (NB_CALCS,),dtype=this_dtype)
print(f"Memory used by pool (GB)     : {mempool.used_bytes()/1e9}")

Memory used by pool (GB)     : 4.6


Loop Cupy

In [7]:
for T in range(1,OUTER_YEARS-1):
    OL_cp[T][...,_MV] = OL_cp[T-1][...,_MV] * ( 1 + OL_cp[T][...,_RETURN]) + OL_cp[T][...,_TRANSAC]
    OL_cp[T][...,_GUARANTEE] = OL_cp[T-1][...,_GUARANTEE]

    OC_cp[T][...,_CALC1] = (OL_cp[T][...,_MV] + OL_cp[T][...,_GUARANTEE])/2
    OC_cp[T][...,_CALC2] = (OL_cp[T][...,_MV]/1000)**2
    OC_cp[T][...,_CALC3] = OL_cp[T][...,_MV]*( 1 + OL_cp[T][...,_RETURN])**1.5
    OC_cp[T][...,_CALC4] = cp.nan_to_num(OL_cp[T][...,_TRANSAC]/OL_cp[T][...,_MV])
    OC_cp[T][...,_CALC5] = 1 - ( 1 + OL_cp[T][...,_RETURN])**0.5
    OC_cp[T][...,_CALC6] = (OL_cp[T][...,_MV] - OL_cp[T][...,_GUARANTEE])/2
    OC_cp[T][...,_CALC7] = (-OL_cp[T][...,_MV]/1000)**2
    OC_cp[T][...,_CALC8] = OL_cp[T][...,_MV]*( 1 - OL_cp[T][...,_RETURN])**1.5
    OC_cp[T][...,_CALC9] = cp.nan_to_num(OL_cp[T][...,_TRANSAC]/OL_cp[T][...,_MV])
    OC_cp[T][...,_CALC10] = 1 - ( 1 - OL_cp[T][...,_RETURN])**0.5
    OC_cp[T][...,_CALC11] = (OL_cp[T][...,_MV] - OL_cp[T][...,_GUARANTEE])/2
    OC_cp[T][...,_CALC12] = (-OL_cp[T][...,_MV]/1000)**2
    OC_cp[T][...,_CALC13] = OL_cp[T][...,_MV]*( 1 - OL_cp[T][...,_RETURN])**1.5
    OC_cp[T][...,_CALC14] = cp.nan_to_num(OL_cp[T][...,_TRANSAC]/OL_cp[T][...,_MV])
    OC_cp[T][...,_CALC15] = 1 - ( 1 - OL_cp[T][...,_RETURN])**0.5

    if T % 10 == 0:
        OL_cp[T][...,_GUARANTEE] = cp.maximum(OL_cp[T-1][...,_GUARANTEE], OL_cp[T][...,_MV])

    OC_cp[T][...,_CLAIMS] = cp.maximum((OL_cp[T][...,_GUARANTEE] - OL_cp[T][...,_MV]),0)


Summarize Cupy

In [8]:
CP_result = cp.nansum(cp.nanmean(cp.nansum(OC_cp[...,_CLAIMS],axis=0),axis=0),axis=0)

print(CP_result)

1446877381.640971


# =============================================================================
# NUMBA CUDA KERNEL APPROACH
# =============================================================================

In [ ]:
import math
from numba import cuda
import warnings
from numba.core.errors import NumbaPerformanceWarning
warnings.filterwarnings('ignore', category=NumbaPerformanceWarning)

@cuda.jit
def numba_projection_kernel(OL_data, OC_data, n_years, n_scens, n_policies):
    """
    Account-first kernel: chaque thread traite une paire (scénario, police)
    à travers TOUTES les années séquentiellement.

    Avantages:
    - Un seul kernel launch (vs ~98 itérations Python avec CuPy)
    - État (MV, guarantee) reste dans les registres entre timesteps
    - Toutes les opérations fusionnées dans un seul kernel
    """
    # 2D grid: (scenario, policy)
    scn_idx = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    pol_idx = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y

    if scn_idx >= n_scens or pol_idx >= n_policies:
        return

    # Indices des champs
    _MV, _RETURN, _TRANSAC, _GUARANTEE = 3, 4, 5, 6
    _CLAIMS = 0

    # Load initial state
    mv = OL_data[0, scn_idx, pol_idx, _MV]
    guarantee = OL_data[0, scn_idx, pol_idx, _GUARANTEE]

    # Process all years sequentially
    for T in range(1, n_years - 1):
        ret = OL_data[T, scn_idx, pol_idx, _RETURN]
        transac = OL_data[T, scn_idx, pol_idx, _TRANSAC]

        # Update MV
        mv = mv * (1.0 + ret) + transac
        OL_data[T, scn_idx, pol_idx, _MV] = mv
        OL_data[T, scn_idx, pol_idx, _GUARANTEE] = guarantee

        # All calculations
        OC_data[T, scn_idx, pol_idx, 1] = (mv + guarantee) / 2.0
        OC_data[T, scn_idx, pol_idx, 2] = (mv / 1000.0) ** 2
        OC_data[T, scn_idx, pol_idx, 3] = mv * math.pow(1.0 + ret, 1.5)
        OC_data[T, scn_idx, pol_idx, 4] = transac / mv if mv != 0.0 else 0.0
        OC_data[T, scn_idx, pol_idx, 5] = 1.0 - math.pow(1.0 + ret, 0.5)
        OC_data[T, scn_idx, pol_idx, 6] = (mv - guarantee) / 2.0
        OC_data[T, scn_idx, pol_idx, 7] = (-mv / 1000.0) ** 2
        OC_data[T, scn_idx, pol_idx, 8] = mv * math.pow(1.0 - ret, 1.5)
        OC_data[T, scn_idx, pol_idx, 9] = transac / mv if mv != 0.0 else 0.0
        OC_data[T, scn_idx, pol_idx, 10] = 1.0 - math.pow(1.0 - ret, 0.5)
        OC_data[T, scn_idx, pol_idx, 11] = (mv - guarantee) / 2.0
        OC_data[T, scn_idx, pol_idx, 12] = (-mv / 1000.0) ** 2
        OC_data[T, scn_idx, pol_idx, 13] = mv * math.pow(1.0 - ret, 1.5)
        OC_data[T, scn_idx, pol_idx, 14] = transac / mv if mv != 0.0 else 0.0
        OC_data[T, scn_idx, pol_idx, 15] = 1.0 - math.pow(1.0 - ret, 0.5)

        # Guarantee ratchet every 10 years
        if T % 10 == 0:
            guarantee = max(guarantee, mv)
            OL_data[T, scn_idx, pol_idx, _GUARANTEE] = guarantee

        # Claims
        OC_data[T, scn_idx, pol_idx, _CLAIMS] = max(guarantee - mv, 0.0)

In [ ]:
# Load Numba - Transfer data to GPU
OL_numba = cuda.to_device(OL_np.copy())
OC_numba = cuda.device_array((OUTER_YEARS, OUTER_SCENS, NB_POLICIES, NB_CALCS), dtype=np.float64)

print(f"Memory used (GB): {(OL_numba.nbytes + OC_numba.nbytes)/1e9:.2f}")

In [ ]:
# Loop Numba - Single kernel launch!
threads_per_block = (16, 16)
blocks_x = (OUTER_SCENS + 15) // 16
blocks_y = (NB_POLICIES + 15) // 16
grid = (blocks_x, blocks_y)

numba_projection_kernel[grid, threads_per_block](
    OL_numba, OC_numba,
    OUTER_YEARS, OUTER_SCENS, NB_POLICIES
)
cuda.synchronize()

In [ ]:
# Summarize Numba
OC_numba_cp = cp.asarray(OC_numba)
NUMBA_result = cp.nansum(cp.nanmean(cp.nansum(OC_numba_cp[..., _CLAIMS], axis=0), axis=0), axis=0)

print(f"Numba result: {NUMBA_result}")
print(f"CuPy result:  {CP_result}")
print(f"Match: {abs(float(NUMBA_result) - float(CP_result)) < 1e-6}")

In [ ]:
# =============================================================================
# BENCHMARK COMPARISON
# =============================================================================
from datetime import datetime

def benchmark_cupy(OL_np, n_runs=3):
    times = []
    for _ in range(n_runs):
        OL_cp = cp.asarray(OL_np.copy())
        OC_cp = cp.zeros(OL_cp.shape[:-1] + (NB_CALCS,), dtype='float64')
        cp.cuda.Stream.null.synchronize()

        t0 = datetime.now()
        for T in range(1, OUTER_YEARS-1):
            OL_cp[T][...,_MV] = OL_cp[T-1][...,_MV] * (1 + OL_cp[T][...,_RETURN]) + OL_cp[T][...,_TRANSAC]
            OL_cp[T][...,_GUARANTEE] = OL_cp[T-1][...,_GUARANTEE]
            OC_cp[T][...,_CALC1] = (OL_cp[T][...,_MV] + OL_cp[T][...,_GUARANTEE])/2
            OC_cp[T][...,_CALC2] = (OL_cp[T][...,_MV]/1000)**2
            OC_cp[T][...,_CALC3] = OL_cp[T][...,_MV]*(1 + OL_cp[T][...,_RETURN])**1.5
            OC_cp[T][...,_CALC4] = cp.nan_to_num(OL_cp[T][...,_TRANSAC]/OL_cp[T][...,_MV])
            OC_cp[T][...,_CALC5] = 1 - (1 + OL_cp[T][...,_RETURN])**0.5
            OC_cp[T][...,_CALC6] = (OL_cp[T][...,_MV] - OL_cp[T][...,_GUARANTEE])/2
            OC_cp[T][...,_CALC7] = (-OL_cp[T][...,_MV]/1000)**2
            OC_cp[T][...,_CALC8] = OL_cp[T][...,_MV]*(1 - OL_cp[T][...,_RETURN])**1.5
            OC_cp[T][...,_CALC9] = cp.nan_to_num(OL_cp[T][...,_TRANSAC]/OL_cp[T][...,_MV])
            OC_cp[T][...,_CALC10] = 1 - (1 - OL_cp[T][...,_RETURN])**0.5
            OC_cp[T][...,_CALC11] = (OL_cp[T][...,_MV] - OL_cp[T][...,_GUARANTEE])/2
            OC_cp[T][...,_CALC12] = (-OL_cp[T][...,_MV]/1000)**2
            OC_cp[T][...,_CALC13] = OL_cp[T][...,_MV]*(1 - OL_cp[T][...,_RETURN])**1.5
            OC_cp[T][...,_CALC14] = cp.nan_to_num(OL_cp[T][...,_TRANSAC]/OL_cp[T][...,_MV])
            OC_cp[T][...,_CALC15] = 1 - (1 - OL_cp[T][...,_RETURN])**0.5
            if T % 10 == 0:
                OL_cp[T][...,_GUARANTEE] = cp.maximum(OL_cp[T-1][...,_GUARANTEE], OL_cp[T][...,_MV])
            OC_cp[T][...,_CLAIMS] = cp.maximum((OL_cp[T][...,_GUARANTEE] - OL_cp[T][...,_MV]), 0)
        cp.cuda.Stream.null.synchronize()
        times.append((datetime.now() - t0).total_seconds())
        del OL_cp, OC_cp
    return sum(times) / len(times)

def benchmark_numba(OL_np, n_runs=3):
    times = []
    threads_per_block = (16, 16)
    blocks_x = (OUTER_SCENS + 15) // 16
    blocks_y = (NB_POLICIES + 15) // 16
    grid = (blocks_x, blocks_y)

    for _ in range(n_runs):
        d_OL = cuda.to_device(OL_np.copy())
        d_OC = cuda.device_array((OUTER_YEARS, OUTER_SCENS, NB_POLICIES, NB_CALCS), dtype=np.float64)
        cuda.synchronize()

        t0 = datetime.now()
        numba_projection_kernel[grid, threads_per_block](d_OL, d_OC, OUTER_YEARS, OUTER_SCENS, NB_POLICIES)
        cuda.synchronize()
        times.append((datetime.now() - t0).total_seconds())
        del d_OL, d_OC
    return sum(times) / len(times)

# Warm-up
_ = benchmark_cupy(OL_np, n_runs=1)
_ = benchmark_numba(OL_np, n_runs=1)

# Benchmark
cupy_time = benchmark_cupy(OL_np, n_runs=3)
numba_time = benchmark_numba(OL_np, n_runs=3)

print(f"CuPy Loop time:  {cupy_time:.3f}s")
print(f"Numba Loop time: {numba_time:.3f}s")
print(f"Speedup:         {cupy_time/numba_time:.1f}x (Numba faster)")